# Aula 15 - Notebook: Simulação de Injeção de Falhas e Desvio Automático de Fluxo na Linha de Envase

Neste notebook simulamos uma contingência operacional em tempo real na **Linha de Envasamento de Bebidas (SCADA-Core - Grupo 3)**: a perda de estanqueidade e vazamento na tubulação principal de dosagem com acionamento do intertravamento de segurança e recálculo dinâmico da rota pelo algoritmo de Dijkstra com comutação para a linha de bypass `XV_BYPASS`.


In [ ]:
import time
import heapq
from typing import Dict, Any, Tuple, Optional, Set, List

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz: List[List[float]], rotulos_linhas: List[str], rotulos_cols: List[str]) -> str:
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "INF" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float, 
                           tag_valvula: str, diametro_pol: float = 3.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem, "Destino": destino,
            "Comprimento (m)": comprimento_m, "Válvula ISA": tag_valvula, "Diâmetro (pol)": diametro_pol
        })

def criar_rede_envase_padrao() -> GrafoTubulacao:
    nos_envase = [
        "TS1_Suprimento",
        "VS1_Succao",
        "BC1_Bomba",
        "AS1_Acumulador",
        "VS2_Envase",
        "SQ2_Medicao",
        "EST_Envase",
        "VALV_Alivio"
    ]
    g = GrafoTubulacao(nos_envase)
    g.adicionar_tubulacao("TS1_Suprimento", "VS1_Succao", 5.0, "VS1", 3.0)
    g.adicionar_tubulacao("VS1_Succao", "BC1_Bomba", 3.0, "SP1_SQ1", 3.0)
    g.adicionar_tubulacao("BC1_Bomba", "AS1_Acumulador", 8.0, "CHECK_V", 2.5)
    g.adicionar_tubulacao("AS1_Acumulador", "VS2_Envase", 10.0, "VS2", 2.0)
    g.adicionar_tubulacao("AS1_Acumulador", "VALV_Alivio", 6.0, "SP2_VS3", 2.0)
    g.adicionar_tubulacao("VALV_Alivio", "TS1_Suprimento", 12.0, "RET_V", 2.0)
    g.adicionar_tubulacao("VS2_Envase", "SQ2_Medicao", 2.0, "SQ2_IN", 1.5)
    g.adicionar_tubulacao("SQ2_Medicao", "EST_Envase", 1.5, "BICO_FILL", 1.5)
    # Linha de Bypass de Contingencia: 14.0m de percurso contornando a bancada
    g.adicionar_tubulacao("AS1_Acumulador", "SQ2_Medicao", 14.0, "XV_BYPASS", 2.0)
    return g

class RoteadorDijkstra:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def calcular_menor_caminho(self, origem: str, destino: str, 
                               bloqueios: Optional[Set[str]] = None) -> Tuple[float, List[str]]:
        if bloqueios is None: 
            bloqueios = set()
        if origem in bloqueios or destino in bloqueios: 
            return float('inf'), []
            
        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        
        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]: 
                continue
            if u == destino: 
                break
                
            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                v = self.g.idx_to_v[v_idx]
                peso = self.g.adj_pesos[u_idx][v_idx]
                if peso < float('inf') and v not in bloqueios:
                    nova_d = d_u + peso
                    if nova_d < dist[v]:
                        dist[v] = nova_d
                        pred[v] = u
                        heapq.heappush(heap, (nova_d, v))
                        
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        caminho.reverse()
        
        if caminho and caminho[0] == origem:
            return dist[destino], caminho
        return float('inf'), []

class SistemaDesvioAutomatico:
    def __init__(self, grafo: GrafoTubulacao):
        self.grafo = grafo
        self.roteador = RoteadorDijkstra(grafo)

    def tratar_evento_vazamento(self, origem_vaz: str, destino_vaz: str,
                                origem_fluxo: str, destino_fluxo: str) -> Dict[str, Any]:
        t0 = time.perf_counter()
        u = self.grafo.v_to_idx[origem_vaz]
        v = self.grafo.v_to_idx[destino_vaz]
        
        # Punicao topologica instantanea
        self.grafo.adj_pesos[u][v] = float('inf')
        self.grafo.adj_binaria[u][v] = 0
        
        novo_custo, nova_rota = self.roteador.calcular_menor_caminho(origem_fluxo, destino_fluxo)
        t_ms = (time.perf_counter() - t0) * 1000.0
        
        status_acao = "ROTA_DESVIADA_COM_SUCESSO" if novo_custo < float('inf') else "TRIP_EMERGENCIA_SEM_ROTA"
        
        return {
            "Trecho_Isolado": f"{origem_vaz} -> {destino_vaz}",
            "Nova_Rota_Ativa": " -> ".join(nova_rota) if nova_rota else "NENHUMA (PARADA DE SEGURANÇA)",
            "Comprimento_Total_m": f"{novo_custo:.1f}" if novo_custo < float('inf') else "INF",
            "Tempo_Decisao_ms": f"{t_ms:.4f}",
            "Status_SCADA": status_acao
        }

# Execucao e Validacao Experimental dos Cenarios
rede = criar_rede_envase_padrao()
roteador_nom = RoteadorDijkstra(rede)

# 1. Rota Nominal (Sem falhas)
c_nom, r_nom = roteador_nom.calcular_menor_caminho("TS1_Suprimento", "EST_Envase")
print("=== ESTADO NOMINAL DA LINHA DE ENVASE ===")
print(f"Rota Nominal Padrao: {' -> '.join(r_nom)} | Comprimento: {c_nom:.1f} m\n")
assert "VS2_Envase" in r_nom, "A rota nominal padrao deve utilizar a valvula de dosagem VS2!"
assert c_nom == 29.5, f"Comprimento nominal esperado de 29.5m, obtido: {c_nom}"

# 2. Injecao de Falha: Vazamento no duto do Acumulador para a Valvula de Envase (AS1 -> VS2)
desviador = SistemaDesvioAutomatico(rede)
res_vaz1 = desviador.tratar_evento_vazamento("AS1_Acumulador", "VS2_Envase", "TS1_Suprimento", "EST_Envase")

print("=== RELATORIO SCADA: DESVIO AUTOMATICO EM CONTINGENCIA ===")
print(formatar_tabela([res_vaz1]))

# Validacoes formais do Caso 1 (Desvio para Bypass)
assert "SQ2_Medicao" in res_vaz1["Nova_Rota_Ativa"]
assert "VS2_Envase" not in res_vaz1["Nova_Rota_Ativa"]
assert res_vaz1["Status_SCADA"] == "ROTA_DESVIADA_COM_SUCESSO"
assert float(res_vaz1["Comprimento_Total_m"]) == 31.5

# 3. Injecao de Falha Critica Adicional (Ruptura tambem no Bypass -> Perda Total de Rota)
res_vaz2 = desviador.tratar_evento_vazamento("AS1_Acumulador", "SQ2_Medicao", "TS1_Suprimento", "EST_Envase")
print("\n=== RELATORIO SCADA: FALHA CATASTROFICA (TRIP DE EMERGENCIA) ===")
print(formatar_tabela([res_vaz2]))

assert res_vaz2["Status_SCADA"] == "TRIP_EMERGENCIA_SEM_ROTA"
assert res_vaz2["Comprimento_Total_m"] == "INF"
print("\n✓ Todos os testes de injecao de falhas e desvio automatico foram validados com 100% de sucesso!")
